# Use breakpoints effectively

**Problem:** You need to debug a problem in your code and want to use `pdb`
efficiently.

This guide covers practical debugging workflows including conditional breakpoints,
post-mortem debugging, and integrating the debugger into your development process.

**Note:** Because `pdb` is an interactive terminal debugger, debugging sessions
are shown as example output. To follow along, copy the code into `.py` files and
run them from the command line.

## Setting breakpoints with `breakpoint()`

The simplest way to set a breakpoint is to place `breakpoint()` in your source
code. When Python reaches that line, it opens the `pdb` prompt.

```python
def process_item(item: dict) -> dict:
    """Process a single item."""
    breakpoint()  # Execution pauses here
    result = {"name": item["name"].upper(), "value": item["value"] * 2}
    return result
```

This is equivalent to the older `import pdb; pdb.set_trace()` pattern, but
simpler and more flexible.

### Disabling breakpoints without removing them

You can disable all `breakpoint()` calls using the `PYTHONBREAKPOINT` environment
variable:

```bash
# Disable all breakpoints
PYTHONBREAKPOINT=0 python my_script.py

# Use a different debugger (for example, ipdb)
PYTHONBREAKPOINT=ipdb.set_trace python my_script.py
```

This is useful when you want to leave `breakpoint()` calls in your code temporarily
without them triggering in production or during automated testing.

## Conditional breakpoints

When debugging a loop or a function that is called many times, you may only
want to break when a specific condition is met.

### In source code

Use a regular `if` statement:

```python
for i, item in enumerate(items):
    if item["status"] == "error":
        breakpoint()  # Only breaks on error items
    process(item)
```

### In the debugger

Use the `b` (break) command with a condition:

```
(Pdb) b 15, item["status"] == "error"
Breakpoint 1 at script.py:15
(Pdb) c
```

The breakpoint only triggers when the condition evaluates to `True`.

## Essential debugging workflow

Here is a step-by-step workflow for common debugging scenarios.

### The n/s/c/p pattern

Most debugging sessions use just four commands:

1. **`p variable`** -- Inspect the current state
2. **`n`** -- Step over the current line
3. **`s`** -- Step into a function to see what happens inside
4. **`c`** -- Continue to the next breakpoint or end of program

### Example session

```
$ python -m pdb my_script.py
> my_script.py(1)<module>()
-> import logging
(Pdb) b process_order       # Set breakpoint at function
Breakpoint 1 at my_script.py:10
(Pdb) c                      # Continue to the breakpoint
> my_script.py(11)process_order()
-> subtotal = 0.0
(Pdb) p items                # Inspect the arguments
[{'name': 'Widget', 'price': 9.99, 'quantity': 3}]
(Pdb) n                      # Step to next line
> my_script.py(12)process_order()
-> for item in items:
(Pdb) n                      # Step into the loop
> my_script.py(13)process_order()
-> item_total = item['price'] * item['quantity']
(Pdb) n                      # Execute the calculation
> my_script.py(14)process_order()
-> subtotal += item_total
(Pdb) p item_total           # Check the result
29.97
(Pdb) c                      # Continue to completion
```

## Post-mortem debugging

Post-mortem debugging lets you inspect the state of your program at the point
where an unhandled exception occurred.

In [ ]:
def parse_config(config_text: str) -> dict:
    """Parse a simple key=value configuration string.

    Args:
        config_text: A string of key=value pairs, one per line.

    Returns:
        A dictionary of configuration values.
    """
    config = {}
    for line in config_text.strip().split("\n"):
        key, value = line.split("=")
        config[key.strip()] = value.strip()
    return config


# This works
good_config = "host=localhost\nport=8080"
print(parse_config(good_config))

# This will fail (line without '=')
bad_config = "host=localhost\nthis line has no equals sign\nport=8080"
try:
    parse_config(bad_config)
except ValueError as exc:
    print("Caught error:", exc)

To debug the error, save the code without the `try`/`except` and run:

```bash
python -m pdb config_parser.py
```

When the `ValueError` occurs, `pdb` opens automatically:

```
ValueError: not enough values to unpack (expected 2, got 1)
Uncaught exception. Entering post mortem debugging
> config_parser.py(9)parse_config()
-> key, value = line.split("=")
(Pdb) p line
'this line has no equals sign'
(Pdb) p line.split("=")
['this line has no equals sign']
```

Now you can see that the problematic line does not contain an `=` sign, so
`split("=")` returns only one item.

## Integrating with your development process

### When to use each debugging technique

| Technique | Best for |
|-----------|----------|
| `print()` | Quick checks in simple scripts |
| `logging` | Persistent diagnostics, production monitoring |
| `breakpoint()` | Interactive investigation of complex bugs |
| `python -m pdb` | Debugging crashes and unhandled exceptions |

### A recommended debugging approach

1. **Reproduce the bug** with a minimal test case
2. **Check log output** to narrow down the problem area
3. **Add a breakpoint** near the suspected cause
4. **Inspect and step** through the code to verify your hypothesis
5. **Fix the bug** and write a test to prevent regression
6. **Remove the breakpoint** and verify the fix

## Quick reference: useful `pdb` commands

| Command | Description |
|---------|-------------|
| `n` | Execute current line (step over) |
| `s` | Step into a function call |
| `c` | Continue to next breakpoint |
| `r` | Continue until the current function returns |
| `p expr` | Print the value of an expression |
| `pp expr` | Pretty-print the value |
| `l` | List source code around the current line |
| `ll` | List the entire current function |
| `w` | Show the call stack |
| `u` | Move up one level in the call stack |
| `d` | Move down one level |
| `b N` | Set a breakpoint at line N |
| `b N, cond` | Set a conditional breakpoint |
| `cl N` | Clear breakpoint number N |
| `q` | Quit the debugger |